# 06 — Carregamento de imagens

Widgets de upload e conversão de ficheiros para arrays NumPy.

**Dependências:** `numpy`, `matplotlib`, `ipywidgets`, `IPython`.

**Pré-requisito:** executar `01_functions.ipynb` se usar `preprocessar=True`.


## Importações


In [ ]:
import io

import ipywidgets as widgets
import matplotlib.image as mpimg
import numpy as np
from IPython.display import display


## Pré-processamento no carregamento

As funções abaixo assumem que `garantir_uint8` e `converter_para_grayscale` estão definidas (executar `01_functions.ipynb` antes).


In [ ]:
def _extrair_ficheiro_upload(upload_widget):
    """Extrai (nome, bytes) do valor do FileUpload."""
    if not upload_widget.value:
        return None, None
    valor = upload_widget.value
    if isinstance(valor, dict):
        ficheiro = next(iter(valor.values()))
    elif isinstance(valor, (list, tuple)):
        ficheiro = valor[0]
    else:
        return None, None
    return ficheiro.get("name"), ficheiro.get("content")


## Criação de widgets


In [ ]:
def criar_uploader(descricao="Upload", aceitar="image/*"):
    """Cria e apresenta um FileUpload único."""
    uploader = widgets.FileUpload(
        accept=aceitar,
        multiple=False,
        description=descricao,
    )
    display(uploader)
    return uploader


def criar_uploaders_imagem_mascara():
    """Dois uploaders: imagem principal e máscara."""
    upload_img = widgets.FileUpload(accept="image/*", multiple=False, description="Imagem")
    upload_mask = widgets.FileUpload(accept="image/*", multiple=False, description="Máscara")
    display(upload_img)
    display(upload_mask)
    return upload_img, upload_mask


def criar_uploaders_multiplos(descricoes):
    """Lista de uploaders com descrições personalizadas."""
    uploaders = []
    for desc in descricoes:
        w = widgets.FileUpload(accept="image/*", multiple=False, description=desc)
        display(w)
        uploaders.append(w)
    return uploaders


## Carregamento


In [ ]:
def carregar_de_upload(upload_widget, preprocessar=False, converter_gray=False):
    """
    Carrega imagem do widget.

    preprocessar: garantir_uint8
    converter_gray: converter_para_grayscale
    """
    nome, conteudo = _extrair_ficheiro_upload(upload_widget)
    if conteudo is None:
        return None, None
    img = mpimg.imread(io.BytesIO(conteudo))
    if preprocessar:
        img = garantir_uint8(img)
    if converter_gray:
        img = converter_para_grayscale(img)
    return img, nome


def carregar_imagem_e_mascara(upload_img, upload_mask, validar=True):
    """
    Carrega par imagem/máscara com validações opcionais.
    Requer funções de 01-functions.
    """
    img, nome_img = carregar_de_upload(upload_img, preprocessar=True, converter_gray=True)
    mask_raw, nome_mask = carregar_de_upload(upload_mask, preprocessar=True, converter_gray=True)
    if img is None or mask_raw is None:
        return None, None, None, None
    if validar:
        if not validar_dimensoes(img, mask_raw):
            raise ValueError("Dimensões incompatíveis entre imagem e máscara.")
        ok, mask = validar_mascara_binaria(mask_raw)
        if not ok:
            raise ValueError("Máscara com demasiados valores não binários.")
    else:
        mask = mask_raw
    return img, mask, nome_img, nome_mask


## Utilização típica

```python
%run 01_functions.ipynb
u_img, u_mask = criar_uploaders_imagem_mascara()
# Selecionar ficheiros no browser, depois:
img, mask, _, _ = carregar_imagem_e_mascara(u_img, u_mask)
```
